1 importy </br>
2 wczytanie splitów</br>
3 filtr young / old</br>
4 tf.data dataset</br>
5 generator</br>
6 discriminator</br>
7 loss functions</br>
8 training loop</br>

In [3]:
import pandas as pd

train_df = pd.read_csv("../data/metadata/train.csv")
val_df = pd.read_csv("../data/metadata/val.csv")
test_df = pd.read_csv("../data/metadata/test.csv")

In [1]:
import tensorflow as tf

IMG_SIZE = 256
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

print("TensorFlow:", tf.__version__)

TensorFlow: 2.21.0


Load the splits

In [4]:
#load the splits
young_df = train_df[train_df["age_group"] == "young"]
old_df = train_df[train_df["age_group"] == "old"]

Image Loader

In [5]:
def load_image(path):

    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)

    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))

    img = (tf.cast(img, tf.float32) / 127.5) - 1
            #for gan we use [-1, 1] range

    return img

Dataset

In [6]:
young_dataset = tf.data.Dataset.from_tensor_slices(young_df["image_path"])
young_dataset = young_dataset.map(load_image).batch(1)

old_dataset = tf.data.Dataset.from_tensor_slices(old_df["image_path"])
old_dataset = old_dataset.map(load_image).batch(1)

Archtecture is:</br>
</br>
G : young → old </br>
F : old → young</br>
</br>
D_Y : discriminator young</br>
D_O : discriminator old</br>

Downsample

In [7]:

from tensorflow.keras import layers

def downsample(filters, size, apply_instancenorm=True):

    initializer = tf.random_normal_initializer(0., 0.02)

    result = tf.keras.Sequential()
    result.add(
        layers.Conv2D(filters, size, strides=2, padding="same",
                      kernel_initializer=initializer, use_bias=False)
    )

    if apply_instancenorm:
        result.add(layers.BatchNormalization())

    result.add(layers.LeakyReLU())

    return result

Upsample

In [8]:
def upsample(filters, size):

    initializer = tf.random_normal_initializer(0., 0.02)

    result = tf.keras.Sequential()
    result.add(
        layers.Conv2DTranspose(filters, size, strides=2,
                               padding="same",
                               kernel_initializer=initializer,
                               use_bias=False)
    )

    result.add(layers.BatchNormalization())
    result.add(layers.ReLU())

    return result

Generator model

In [9]:
def build_generator():

    inputs = layers.Input(shape=[256,256,3])

    x = downsample(64,4,apply_instancenorm=False)(inputs)
    x = downsample(128,4)(x)
    x = downsample(256,4)(x)

    x = upsample(128,4)(x)
    x = upsample(64,4)(x)

    outputs = layers.Conv2D(
        3,4,
        strides=1,
        padding="same",
        activation="tanh"
    )(x)

    return tf.keras.Model(inputs, outputs)

Create 2 generators

G : young → old </br>
F : old → young

In [ ]:
generator_g = build_generator()
generator_f = build_generator()

Discriminator model

In [ ]:
def build_discriminator():

    initializer = tf.random_normal_initializer(0.,0.02)

    inp = layers.Input(shape=[256,256,3])

    x = downsample(64,4,False)(inp)
    x = downsample(128,4)(x)
    x = downsample(256,4)(x)

    x = layers.Conv2D(512,4,strides=1,padding="same",
                      kernel_initializer=initializer)(x)

    x = layers.LeakyReLU()(x)

    x = layers.Conv2D(1,4,strides=1,padding="same",
                      kernel_initializer=initializer)(x)

    return tf.keras.Model(inp,x)

Create 2 discriminators

In [ ]:
discriminator_x = build_discriminator()
discriminator_y = build_discriminator()

## Loss functions

binery creossentropy


In [12]:
loss_obj = tf.keras.losses.BinaryCrossentropy(from_logits=True)

In [13]:
def discriminator_loss(real, generated):

    real_loss = loss_obj(tf.ones_like(real), real)
    generated_loss = loss_obj(tf.zeros_like(generated), generated)

    total_disc_loss = real_loss + generated_loss

    return total_disc_loss * 0.5

Generator adversial loss

In [14]:
def generator_loss(generated):

    return loss_obj(tf.ones_like(generated), generated)

Cycle consistency loss

In [15]:
LAMBDA = 10

In [16]:
def cycle_loss(real_image, cycled_image):

    loss = tf.reduce_mean(tf.abs(real_image - cycled_image))

    return LAMBDA * loss

Identity loss

In [17]:
def identity_loss(real_image, same_image):

    loss = tf.reduce_mean(tf.abs(real_image - same_image))

    return LAMBDA * 0.5 * loss

Optimizers

In [18]:
generator_g_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)
generator_f_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)

discriminator_x_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)
discriminator_y_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)

Training step

In [ ]:
@tf.function
def train_step(real_x, real_y):

    with tf.GradientTape(persistent=True) as tape:

        fake_y = generator_g(real_x, training=True)
        cycled_x = generator_f(fake_y, training=True)

        fake_x = generator_f(real_y, training=True)
        cycled_y = generator_g(fake_x, training=True)

        same_x = generator_f(real_x, training=True)
        same_y = generator_g(real_y, training=True)

        disc_real_x = discriminator_x(real_x, training=True)
        disc_real_y = discriminator_y(real_y, training=True)

        disc_fake_x = discriminator_x(fake_x, training=True)
        disc_fake_y = discriminator_y(fake_y, training=True)

        gen_g_loss = generator_loss(disc_fake_y)
        gen_f_loss = generator_loss(disc_fake_x)

        total_cycle_loss = cycle_loss(real_x, cycled_x) + cycle_loss(real_y, cycled_y)

        total_gen_g_loss = gen_g_loss + total_cycle_loss + identity_loss(real_y, same_y)
        total_gen_f_loss = gen_f_loss + total_cycle_loss + identity_loss(real_x, same_x)

        disc_x_loss = discriminator_loss(disc_real_x, disc_fake_x)
        disc_y_loss = discriminator_loss(disc_real_y, disc_fake_y)

    generator_g_gradients = tape.gradient(
        total_gen_g_loss,
        generator_g.trainable_variables
    )

    generator_f_gradients = tape.gradient(
        total_gen_f_loss,
        generator_f.trainable_variables
    )

    discriminator_x_gradients = tape.gradient(
        disc_x_loss,
        discriminator_x.trainable_variables
    )

    discriminator_y_gradients = tape.gradient(
        disc_y_loss,
        discriminator_y.trainable_variables
    )

    generator_g_optimizer.apply_gradients(
        zip(generator_g_gradients, generator_g.trainable_variables)
    )

    generator_f_optimizer.apply_gradients(
        zip(generator_f_gradients, generator_f.trainable_variables)
    )

    discriminator_x_optimizer.apply_gradients(
        zip(discriminator_x_gradients, discriminator_x.trainable_variables)
    )

    discriminator_y_optimizer.apply_gradients(
        zip(discriminator_y_gradients, discriminator_y.trainable_variables)
    )

Training loop

In [ ]:
EPOCHS = 100

for epoch in range(EPOCHS):

    for image_x, image_y in tf.data.Dataset.zip((young_dataset, old_dataset)):
        train_step(image_x, image_y)

    print("Epoch completed:", epoch)